<a href="https://colab.research.google.com/github/pragatiagarwal802-wq/surat-ev-infrastructure-analysis/blob/main/SEARCHENGINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Setup
!pip install h3 pandas scikit-learn

import pandas as pd
import numpy as np
import h3
import json
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

class SuratEVFullPipeline:
    def __init__(self, file_path):
        self.df = pd.read_csv(file_path)
        self.scaler = MinMaxScaler()
        print(f"🚀 Loaded {len(self.df)} sites.")

    def process_and_convert(self, weights, k_hotspots=10):
        # --- PART 1: SCORE CALCULATION ---
        layers = ['population_5km_buffer', 'traffic_density_score', 'grid_score', 'competitor_count', 'flood_risk']
        for col in layers:
            if col in self.df.columns:
                self.df[f'norm_{col}'] = self.scaler.fit_transform(self.df[[col]])
            else:
                self.df[f'norm_{col}'] = 0

        self.df['readiness_score'] = (
            self.df['norm_population_5km_buffer'] * weights.get('pop', 0.4) +
            self.df['norm_traffic_density_score'] * weights.get('traffic', 0.3) +
            self.df['norm_grid_score'] * weights.get('grid', 0.2) +
            (self.df['norm_competitor_count'] * weights.get('comp', -0.05)) +
            (self.df['norm_flood_risk'] * weights.get('flood', -0.05))
        )
        self.df['readiness_score'] = self.df['readiness_score'].clip(0, 1) * 100

        # --- PART 2: STRATEGIC TIERING ---
        def get_tier(row):
            if row['readiness_score'] > 75: return "Tier 1: Mega Hub"
            elif row['readiness_score'] > 50: return "Tier 2: Standard Station"
            else: return "Tier 3: Local Point"
        self.df['site_tier'] = self.df.apply(get_tier, axis=1)

        # --- PART 3: SPATIAL HOTSPOTS ---
        top_sites = self.df.nlargest(1000, 'readiness_score').copy()
        top_sites['lat'] = top_sites['hex_id'].apply(lambda x: h3.cell_to_latlng(x)[0])
        top_sites['lng'] = top_sites['hex_id'].apply(lambda x: h3.cell_to_latlng(x)[1])

        kmeans = KMeans(n_clusters=k_hotspots, n_init=10, random_state=42)
        top_sites['cluster'] = kmeans.fit_predict(top_sites[['lat', 'lng']])
        self.df['is_hotspot'] = self.df['hex_id'].isin(top_sites['hex_id'])

        # --- PART 4: GEOJSON CONVERSION (For the Map) ---
        features = []
        # We convert the top 2000 sites for the interactive map (better performance)
        map_subset = self.df.nlargest(2000, 'readiness_score')

        for _, row in map_subset.iterrows():
            boundary = h3.cell_to_boundary(row['hex_id'])
            # Flip to [lng, lat] for GeoJSON standard
            coords = [[p[1], p[0]] for p in boundary]
            coords.append(coords[0]) # Close polygon

            features.append({
                "type": "Feature",
                "geometry": {"type": "Polygon", "coordinates": [coords]},
                "properties": {
                    "hex_id": row['hex_id'],
                    "score": round(row['readiness_score'], 1),
                    "tier": row['site_tier'],
                    "hotspot": bool(row['is_hotspot']),
                    "pop": int(row['population_5km_buffer']),
                    "grid": round(row['grid_score'], 1)
                }
            })

        geojson = {"type": "FeatureCollection", "features": features}

        # Save both files
        self.df.to_csv('SURAT_EV_DATA.csv', index=False)
        with open('SURAT_EV_MAP.geojson', 'w') as f:
            json.dump(geojson, f)

        print("✅ SUCCESS: 'SURAT_EV_DATA.csv' and 'SURAT_EV_MAP.geojson' are ready!")

# EXECUTE
my_weights = {'pop': 0.40, 'traffic': 0.30, 'grid': 0.20, 'comp': -0.05, 'flood': -0.05}
pipeline = SuratEVFullPipeline('SURAT_EV_MASTER_COMPLETE.csv')
pipeline.process_and_convert(my_weights)

🚀 Loaded 45881 sites.
✅ SUCCESS: 'SURAT_EV_DATA.csv' and 'SURAT_EV_MAP.geojson' are ready!


In [ ]:
import json
import h3

# 1. Load your final results
df_final = pd.read_csv('/content/SURAT_EV_MASTER_COMPLETE.csv')

# 2. Function to turn a Hex ID into a Map Shape
def make_geojson(df):
    features = []
    for _, row in df.iterrows():
        # Get the 6 corners of the hexagon
        # Using the new H3 v4 'cell_to_boundary'
        points = h3.cell_to_boundary(row['hex_id'])

        # Format it for the Map (Longitude first, then Latitude)
        flipped_points = [[p[1], p[0]] for p in points]
        flipped_points.append(flipped_points[0]) # Close the loop

        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [flipped_points]
            },
            "properties": {
                "id": row['hex_id'],
                "score": round(row['readiness_score'], 2),
                "tier": row['site_tier'],
                "is_hub": bool(row['is_hotspot']),
                "pop": row['population_5km_buffer']
            }
        }
        features.append(feature)

    return {"type": "FeatureCollection", "features": features}

# 3. Save the file
geojson_data = make_geojson(df_final.head(1000)) # We take top 1000 for speed
with open('SURAT_EV_MAP.geojson', 'w') as f:
    json.dump(geojson_data, f)

print("🎉 DONE! Download 'SURAT_EV_MAP.geojson' and send it to her.")

KeyError: 'readiness_score'